# 03 — Sentiment & Emotion Labeling
**Reddit Opinion & Community Research — Pipeline step 3 of 11**


In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("pandas", None), ("pyarrow", None), ("numpy", None),
                 ("scikit-learn", "sklearn"), ("langid", None),
                 ("torch", None), ("transformers", None)]:
    ensure(pkg, imp)

print("Packages OK.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Packages OK.


In [2]:
import os, json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

CONTRACT_PATH = os.path.join(BASE_DIR, "processed_v2", "raw_enrichment_contract.json")
with open(CONTRACT_PATH, "r", encoding="utf-8") as f:
    contract = json.load(f)
PROCESSED_V2_DIR = contract["processed_v2_dir"]

GOLD_PATH = os.path.join(BASE_DIR, "annotation", "gold_labels_consensus.parquet")
print("FOUND" if os.path.isfile(GOLD_PATH) else "MISSING", "->", GOLD_PATH)


Mounted at /content/drive
FOUND -> /content/drive/MyDrive/PhD/reddit_opinion_research/annotation/gold_labels_consensus.parquet


## Load, merge, and apply the Fear -> Sadness emotion merge

In [3]:
text_corpus = pd.read_parquet(contract["paths"]["text_corpus"])
gold = pd.read_parquet(GOLD_PATH)

gold["emotion"] = gold["emotion"].replace({"Fear": "Sadness"})

print(f"text_corpus: {len(text_corpus):,} documents")
print(f"gold: {len(gold):,} documents")

overlap = set(gold["doc_id"]) & set(text_corpus["doc_id"])
print(f"Gold doc_ids found in text_corpus: {len(overlap):,} / {len(gold):,}")

text_corpus = text_corpus.merge(
    gold[["doc_id", "language_tag", "sentiment", "emotion"]].rename(
        columns={"sentiment": "gold_sentiment", "emotion": "gold_emotion"}
    ),
    on="doc_id", how="left"
)
text_corpus["is_gold"] = text_corpus["gold_sentiment"].notna()
print(f"is_gold flag set on {text_corpus['is_gold'].sum():,} rows")


text_corpus: 210,994 documents
gold: 916 documents
Gold doc_ids found in text_corpus: 916 / 916
is_gold flag set on 916 rows


In [4]:
print("Sentiment class counts (gold, post-merge):")
print(gold["sentiment"].value_counts())
print("\nEmotion class counts (gold, after Fear -> Sadness merge):")
print(gold["emotion"].value_counts())
print("\nLanguage tag counts (gold, manually assigned):")
print(gold["language_tag"].value_counts())


Sentiment class counts (gold, post-merge):
sentiment
Negative    374
Neutral     332
Positive    210
Name: count, dtype: int64

Emotion class counts (gold, after Fear -> Sadness merge):
emotion
Neutral     360
Anger       250
Joy         179
Sadness      86
Surprise     41
Name: count, dtype: int64

Language tag counts (gold, manually assigned):
language_tag
en    718
ar     86
dl     83
fr     29
Name: count, dtype: int64


## Full-corpus language-ID pass




In [5]:
import langid
langid.set_languages(["en", "fr", "ar"])

def detect_lang(text):
    if not text or not isinstance(text, str):
        return None
    try:
        lang, _ = langid.classify(text)
        return lang
    except Exception:
        return None

print("Running langid over the full corpus (CPU, no GPU needed for this step)...")
text_corpus["langid_pred"] = text_corpus["text"].apply(detect_lang)
print("\nFull-corpus automatic language distribution:")
print(text_corpus["langid_pred"].value_counts(normalize=True).mul(100).round(1))


Running langid over the full corpus (CPU, no GPU needed for this step)...

Full-corpus automatic language distribution:
langid_pred
en    87.5
ar     9.7
fr     2.8
Name: proportion, dtype: float64


In [6]:
gold_dl = text_corpus.loc[text_corpus["language_tag"] == "dl"]
if len(gold_dl):
    print(f"Of {len(gold_dl)} gold documents manually tagged 'dl' (Algerian dialect), langid predicts:")
    print(gold_dl["langid_pred"].value_counts())
    print("\n-> This is the expected miscategorization rate, not a bug: langid has no 'dl' label to assign.")
else:
    print("No gold documents tagged 'dl' found in the merge -- check the language_tag values above.")


Of 83 gold documents manually tagged 'dl' (Algerian dialect), langid predicts:
langid_pred
en    68
fr    10
ar     5
Name: count, dtype: int64

-> This is the expected miscategorization rate, not a bug: langid has no 'dl' label to assign.


## Embed text with DziriBERT 

In [7]:
import torch
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

MODEL_NAME = "alger-ia/dziribert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
if DEVICE == "cuda":
    model = model.half()

@torch.no_grad()
def embed_texts(texts, batch_size=128, max_length=128):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = [t if isinstance(t, str) and t else " " for t in texts[i:i + batch_size]]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        token_embeds = out.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).to(token_embeds.dtype)
        summed = (token_embeds * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        mean_pooled = (summed / counts).float().cpu().numpy()
        all_embeds.append(mean_pooled)
    return np.concatenate(all_embeds, axis=0)

print("DziriBERT loaded. Hidden size:", model.config.hidden_size)


Using device: cuda


config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/446k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DziriBERT loaded. Hidden size: 768


In [8]:
gold_rows = text_corpus[text_corpus["is_gold"]].reset_index(drop=True)
gold_embeddings = embed_texts(gold_rows["text"].tolist(), batch_size=64, max_length=128)
print("gold_embeddings shape:", gold_embeddings.shape)


gold_embeddings shape: (916, 768)


## 5-fold stratified cross-validation 

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

def run_cv(X, y, label_name, n_splits=5):
    min_class_count = pd.Series(y).value_counts().min()
    n_splits = min(n_splits, min_class_count)
    if n_splits < 2:
        print(f"[{label_name}] Smallest class has only {min_class_count} example(s) -- skipping CV, will still fit a final model.")
        return None
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs, f1s = [], []
    for train_idx, test_idx in skf.split(X, y):
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
        clf.fit(X[train_idx], np.array(y)[train_idx])
        pred = clf.predict(X[test_idx])
        accs.append(accuracy_score(np.array(y)[test_idx], pred))
        f1s.append(f1_score(np.array(y)[test_idx], pred, average="macro"))
    print(f"[{label_name}] {n_splits}-fold CV: accuracy = {np.mean(accs):.3f} +/- {np.std(accs):.3f}, "
          f"macro-F1 = {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")
    return {"accuracy_mean": float(np.mean(accs)), "accuracy_std": float(np.std(accs)),
            "macro_f1_mean": float(np.mean(f1s)), "macro_f1_std": float(np.std(f1s)), "n_splits": n_splits}

sentiment_cv = run_cv(gold_embeddings, gold_rows["gold_sentiment"].values, "sentiment")
emotion_cv = run_cv(gold_embeddings, gold_rows["gold_emotion"].values, "emotion")


[sentiment] 5-fold CV: accuracy = 0.605 +/- 0.009, macro-F1 = 0.594 +/- 0.011
[emotion] 5-fold CV: accuracy = 0.548 +/- 0.033, macro-F1 = 0.476 +/- 0.041


In [10]:
# Final classifiers, trained on ALL 916 gold documents (CV above is only for the honest performance estimate)
sentiment_clf = LogisticRegression(max_iter=2000, class_weight="balanced")
sentiment_clf.fit(gold_embeddings, gold_rows["gold_sentiment"].values)

emotion_clf = LogisticRegression(max_iter=2000, class_weight="balanced")
emotion_clf.fit(gold_embeddings, gold_rows["gold_emotion"].values)

print("Final classifiers fit on all", len(gold_rows), "gold documents.")


Final classifiers fit on all 916 gold documents.


## Embed the full corpus

This is the slow step. Chunked to disk so a session disconnect only costs the current chunk, not everything before it — re-running this cell skips chunks that already exist.


In [11]:
CHUNK_SIZE = 20000
BATCH_SIZE = 64
MAX_LENGTH = 128

chunks_dir = os.path.join(PROCESSED_V2_DIR, "embedding_chunks")
os.makedirs(chunks_dir, exist_ok=True)

texts = text_corpus["text"].tolist()
n_total = len(texts)
n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"{n_total:,} documents -> {n_chunks} chunks of up to {CHUNK_SIZE:,}")

for chunk_idx in range(n_chunks):
    chunk_path = os.path.join(chunks_dir, f"chunk_{chunk_idx:04d}.npy")
    if os.path.isfile(chunk_path):
        print(f"Chunk {chunk_idx + 1}/{n_chunks} already done, skipping.")
        continue
    start, end = chunk_idx * CHUNK_SIZE, min((chunk_idx + 1) * CHUNK_SIZE, n_total)
    chunk_embeds = embed_texts(texts[start:end], batch_size=BATCH_SIZE, max_length=MAX_LENGTH)
    np.save(chunk_path, chunk_embeds.astype("float16"))
    print(f"Chunk {chunk_idx + 1}/{n_chunks} done ({end:,}/{n_total:,} docs) -> {chunk_path}")

print("All chunks complete.")


210,994 documents -> 11 chunks of up to 20,000
Chunk 1/11 done (20,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_0000.npy
Chunk 2/11 done (40,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_0001.npy
Chunk 3/11 done (60,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_0002.npy
Chunk 4/11 done (80,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_0003.npy
Chunk 5/11 done (100,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_0004.npy
Chunk 6/11 done (120,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_0005.npy
Chunk 7/11 done (140,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/embedding_chunks/chunk_

In [12]:
all_chunks = [np.load(os.path.join(chunks_dir, f"chunk_{i:04d}.npy")) for i in range(n_chunks)]
full_embeddings = np.concatenate(all_chunks, axis=0)
assert full_embeddings.shape[0] == n_total, "Embedding count doesn't match corpus size -- check for a missing/corrupt chunk."
print("full_embeddings shape:", full_embeddings.shape)

embeddings_path = os.path.join(PROCESSED_V2_DIR, "dziribert_embeddings.npy")
np.save(embeddings_path, full_embeddings)
print("Saved:", embeddings_path, "-- reuse this directly as HOD-Mamba's x_t semantic input feature.")


full_embeddings shape: (210994, 768)
Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/dziribert_embeddings.npy -- reuse this directly as HOD-Mamba's x_t semantic input feature.


## Apply the classifiers to the full corpus

In [13]:
text_corpus["sentiment_label"] = sentiment_clf.predict(full_embeddings)
text_corpus["emotion_label"] = emotion_clf.predict(full_embeddings)

# Gold rows keep their human/LLM-consensus label, not the classifier's own prediction on itself
text_corpus.loc[text_corpus["is_gold"], "sentiment_label"] = text_corpus.loc[text_corpus["is_gold"], "gold_sentiment"]
text_corpus.loc[text_corpus["is_gold"], "emotion_label"] = text_corpus.loc[text_corpus["is_gold"], "gold_emotion"]

print("Full-corpus predicted sentiment distribution:")
print(text_corpus["sentiment_label"].value_counts(normalize=True).mul(100).round(1))
print("\nFull-corpus predicted emotion distribution:")
print(text_corpus["emotion_label"].value_counts(normalize=True).mul(100).round(1))


Full-corpus predicted sentiment distribution:
sentiment_label
Negative    42.9
Neutral     33.9
Positive    23.3
Name: proportion, dtype: float64

Full-corpus predicted emotion distribution:
emotion_label
Neutral     34.3
Anger       27.8
Joy         18.5
Sadness     14.1
Surprise     5.3
Name: proportion, dtype: float64


In [14]:
labeled_path = os.path.join(PROCESSED_V2_DIR, "text_corpus_labeled.parquet")
text_corpus.to_parquet(labeled_path, engine="pyarrow", index=False)
print("Saved:", labeled_path)


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/text_corpus_labeled.parquet


## Write the labeling contract

In [15]:
labeling_contract = {
    "gold_path": GOLD_PATH,
    "paths": {
        "text_corpus_labeled": labeled_path,
        "dziribert_embeddings": embeddings_path,
    },
    "model": "alger-ia/dziribert",
    "gold_class_counts": {
        "sentiment": {str(k): int(v) for k, v in gold["sentiment"].value_counts().items()},
        "emotion": {str(k): int(v) for k, v in gold["emotion"].value_counts().items()},
        "language_tag": {str(k): int(v) for k, v in gold["language_tag"].value_counts().items()},
    },
    "cv_results": {"sentiment": sentiment_cv, "emotion": emotion_cv},
    "full_corpus_language_id_distribution": {
        str(k): float(v) for k, v in text_corpus["langid_pred"].value_counts(normalize=True).items()
    },
    "full_corpus_predicted_distribution": {
        "sentiment": {str(k): int(v) for k, v in text_corpus["sentiment_label"].value_counts().items()},
        "emotion": {str(k): int(v) for k, v in text_corpus["emotion_label"].value_counts().items()},
    },
}

labeling_contract_path = os.path.join(PROCESSED_V2_DIR, "labeling_contract.json")
with open(labeling_contract_path, "w", encoding="utf-8") as f:
    json.dump(labeling_contract, f, indent=2, ensure_ascii=False, default=str)

print("Saved:", labeling_contract_path)
print(json.dumps({"cv_results": labeling_contract["cv_results"]}, indent=2, ensure_ascii=False))


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/labeling_contract.json
{
  "cv_results": {
    "sentiment": {
      "accuracy_mean": 0.6048051793775244,
      "accuracy_std": 0.009417263366559054,
      "macro_f1_mean": 0.5935760810386145,
      "macro_f1_std": 0.01110494258325271,
      "n_splits": 5
    },
    "emotion": {
      "accuracy_mean": 0.5480577334283678,
      "accuracy_std": 0.03287847975296422,
      "macro_f1_mean": 0.4755299735030527,
      "macro_f1_std": 0.04059347190460663,
      "n_splits": 5
    }
  }
}


In [16]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for task, y_col in [("sentiment", "gold_sentiment"), ("emotion", "gold_emotion")]:
    y = gold_rows[y_col].values
    oof_preds = cross_val_predict(
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        gold_embeddings, y, cv=skf
    )
    print(f"\n=== {task} — out-of-fold performance by language_tag ===")
    for lang in gold_rows["language_tag"].unique():
        mask = (gold_rows["language_tag"] == lang).values
        if mask.sum() < 10:
            print(f"  {lang}: n={mask.sum()} (too few for a reliable estimate)")
            continue
        acc = accuracy_score(y[mask], oof_preds[mask])
        f1 = f1_score(y[mask], oof_preds[mask], average="macro", zero_division=0)
        print(f"  {lang}: n={mask.sum()}, accuracy={acc:.3f}, macro_f1={f1:.3f}")


=== sentiment — out-of-fold performance by language_tag ===
  en: n=718, accuracy=0.599, macro_f1=0.591
  dl: n=83, accuracy=0.651, macro_f1=0.640
  ar: n=86, accuracy=0.593, macro_f1=0.539
  fr: n=29, accuracy=0.655, macro_f1=0.640

=== emotion — out-of-fold performance by language_tag ===
  en: n=718, accuracy=0.547, macro_f1=0.475
  dl: n=83, accuracy=0.578, macro_f1=0.501
  ar: n=86, accuracy=0.581, macro_f1=0.495
  fr: n=29, accuracy=0.379, macro_f1=0.237


In [17]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
emotion_oof = cross_val_predict(
    LogisticRegression(max_iter=2000, class_weight="balanced"),
    gold_embeddings, gold_rows["gold_emotion"].values, cv=skf
)

print(classification_report(gold_rows["gold_emotion"].values, emotion_oof, zero_division=0))
print("\nClass counts:")
print(gold_rows["gold_emotion"].value_counts())

              precision    recall  f1-score   support

       Anger       0.60      0.61      0.60       250
         Joy       0.56      0.55      0.56       179
     Neutral       0.62      0.58      0.60       360
     Sadness       0.27      0.35      0.30        86
    Surprise       0.35      0.32      0.33        41

    accuracy                           0.55       916
   macro avg       0.48      0.48      0.48       916
weighted avg       0.56      0.55      0.55       916


Class counts:
gold_emotion
Neutral     360
Anger       250
Joy         179
Sadness      86
Surprise     41
Name: count, dtype: int64
